In [11]:
import pandas as pd
import requests
from datetime import datetime
import calendar

# 1. Construir un dataframe de métricas principales del proyecto y variación en el último mes.

El dataframe resultante tendrá esta forma:

```python
metric, number_today, number_in_last_month
observations, 1521, 23
observers, 124, 2
identifiers, 26, 0
species, 462, 5
```

* number_today = dato actual de observaciones, observadores, identificadores, especies.
* number_in_last_month = dato de fecha actual menos dato registrado 30 días antes (variación en los últimos 30 días).

Usamos llamadas a la API, porque son datos totales. 

Creamos un directorio "data" donde guardaremos todos los csv que vamos a generar. Este dataframe lo guardamos como "data/main_metrics.cvs", sin incluir los índices (index=False).


In [ ]:
# Definimos la función
def get_main_metrics(id_project):
    """
    Obtiene las métricas principales de un proyecto.
    :param id_project: ID del proyecto
    :return: DataFrame con las métricas principales
    """
    # Hacer aquí la función para obtener las métricas principales del proyecto

    return df_main_metrics

In [ ]:
# La utilizamos y guardamos el resultado
df_main_metrics = get_main_metrics(264)

# Aquí deberías haber creado la carpeta 'data' previamente, en este mismo repositorio
df_main_metrics.to_csv('data/main_metrics.csv', index=False)

# 2. Evolución de las métricas principales

Construir un dataframe con esta forma:

```python
month, observations, observers, identifiers, species
2024-01, 185, 15, 7, 62
2024-02, 128, 3, 1, 32
...
```

Los datos no son acumulativos, son del mes en concreto. Lo sacaremos usando llamadas a la API.

Para ello puedes utilizar estas funciones, que te ayudarán a construirlo:

In [21]:
import requests

API_PATH = "https://api.minka-sdg.org/v1"

def get_totals(project_id, year, month, kind="project", session=None):

    if session is None:
        session = requests.Session()

    # Define the API endpoints for the different metrics
    # and construct the URLs with the provided project ID and date range
    if kind == "project":
        url_obs = f"{API_PATH}/observations?project_id={project_id}&month={month}&year={year}"
        url_spe = f"{API_PATH}/observations/species_counts?project_id={project_id}&month={month}&year={year}"
        url_part = f"{API_PATH}/observations/observers?project_id={project_id}&month={month}&year={year}"
        url_ident = f"{API_PATH}/observations/identifiers?project_id={project_id}&month={month}&year={year}"
    elif kind == "place":
        url_obs = f"{API_PATH}/observations?place_id={project_id}&month={month}&year={year}"
        url_spe = f"{API_PATH}/observations/species_counts?place_id={project_id}&month={month}&year={year}"
        url_part = f"{API_PATH}/observations/observers?place_id={project_id}&month={month}&year={year}"
        url_ident = f"{API_PATH}/observations/identifiers?place_id={project_id}&month={month}&year={year}"  

    # Make GET requests to the API endpoints and extract the total results
    total_obs = session.get(url_obs).json()["total_results"]
    total_part = session.get(url_part).json()["total_results"]
    total_ident = session.get(url_ident).json()["total_results"]
    total_spe = session.get(url_spe).json()["total_results"]

    return total_obs, total_part, total_ident, total_spe

In [16]:
from datetime import datetime
import calendar

def get_month_list(years: list) -> dict:
    current_year = datetime.now().year
    current_month = datetime.now().month
    meses = []

    for year in years:
        max_month = 12 if year < current_year else current_month
        for month in range(1, max_month + 1):
            meses.append(f"{year}-{str(month).zfill(2)}")
            
    return meses

In [17]:
meses = get_month_list(range(2022, datetime.now().year + 1))
meses

['2022-01',
 '2022-02',
 '2022-03',
 '2022-04',
 '2022-05',
 '2022-06',
 '2022-07',
 '2022-08',
 '2022-09',
 '2022-10',
 '2022-11',
 '2022-12',
 '2023-01',
 '2023-02',
 '2023-03',
 '2023-04',
 '2023-05',
 '2023-06',
 '2023-07',
 '2023-08',
 '2023-09',
 '2023-10',
 '2023-11',
 '2023-12',
 '2024-01',
 '2024-02',
 '2024-03',
 '2024-04',
 '2024-05',
 '2024-06',
 '2024-07',
 '2024-08',
 '2024-09',
 '2024-10',
 '2024-11',
 '2024-12',
 '2025-01',
 '2025-02',
 '2025-03',
 '2025-04']

In [19]:
# Para cada elemento de la lista, podemos sacar el año y el mes
meses[0].split("-")[0]  # Año
meses[0].split("-")[1]  # Mes

'01'

In [22]:
# Ejemplo de uso con el primer mes

get_totals(
    project_id=264,
    year=int(meses[0].split("-")[0]),
    month=int(meses[0].split("-")[1]),
    kind="project"
)

(0, 0, 0, 0)

Esto nos devuelve un diccionario con los meses como clave y el último día del mes como valor. Así podemos usarlo con la función anterior:

Ahora hay que unir las dos funciones para sacar cada mes y de cada mes sacar los valores de las métricas. Eso nos da los resultados de un mes, que podemos guardar en un diccionario. Y luego unimos los diccionarios de cada mes en una lista de todos los meses. Te pongo debajo un ejemplo de uso con un mes.

In [ ]:
# Ejemplo de proceso con un mes

# Creamos una lista vacía para almacenar los resultados de cada mes
total_metrics = []

# Te enseño cómo construirlo con el primer mes de la lista meses
mes = meses[0]
year = mes.split("-")[0]
month = mes.split("-")[1]
total_obs, total_spe, total_part, total_ident = get_totals(
    project_id=264, year=year, month=month, kind="project"
)

# Creamos un diccionario con los resultados del mes
# y lo añadimos a la lista de métricas totales
total_for_month = {}
total_for_month["month"] = meses[0]
total_for_month["total_obs"] = total_obs
total_for_month["total_spe"] = total_spe
total_for_month["total_part"] = total_part
total_for_month["total_ident"] = total_ident
total_metrics.append(total_for_month)

# Creamos un DataFrame a partir de la lista de métricas totales
df_monthly = pd.DataFrame(total_metrics)

Ahora hay que crear una función que itere por todos los elementos de la lista meses desde el inicio de MINKA y saque los datos para cada mes usando get_totals a un diccionario, los acumule en la lista y la lista la convierta a un dataframe.

Es decir, para cada elemento de los meses, usamos get_totals para sacar las métricas y las almacenamos. Si lo ves complicado lo hacemos juntos.

Ese dataframe lo guardamos como "data/monthly_metrics.csv".

# 3. Taxonomías

Descargamos todas las observaciones del proyecto, usando mecoda_minka. Generamos los dataframes de observaciones y de fotos. A partir de df_obs creamos una función que nos permita ver el número de observaciones por reino, filo, clase... Este rango se tiene que poder indicar como parámetro, para usar la misma función para cualquier rango.

Guardamos df_obs y df_photos como csv en la carpeta `data`. Y no guardamos los índices, como en los casos anteriores.

Creamos la función. Ten en cuenta las columnas de los rangos que tiene el dataframe de df_obs. En las columnas "kingdom", "phylum", "class",... están los rangos taxonómicos superiores a la identificación. El "taxon_name" es el identificado en la observación, y el "taxon_rank" el rango de la identificación. En las otras columnas están los rangos superiores. Esto lo hacemos juntos, que es un poco complicado de explicar por escrito.

In [ ]:
def get_taxon_count(df_obs, rank_level):
    """
    Obtiene el conteo de taxones por nivel de rango.
    :param df_obs: DataFrame con las observaciones
    :param rank_level: Nivel de rango (kingdom, phylum, class, order, family, genus, species)
    :return: DataFrame con el conteo de taxones
    """

    return df_taxon_count

# 4. Especies vistas por primera vez en el proyecto desde el último informe (últimos 30 días)

A partir del df_obs podemos sacar este dato fácilmente. Toma el dataframe, ordénalo por fecha de observación, en orden ascendente (las primeras observaciones estarán más arriba). Ahora quédate solo con las primeras observaciones de cada especie. Es decir:
* Seleccionamos aquellas observaciones que hayan llegado al nivel de especie (columna "taxon_rank" == "species").
* Nos quedamos con la primera observación de cada especie, usando drop_duplicates()
```python
df_first = df_obs.drop_duplicates(subset=["taxon_name"], keep="first")
```
* Así nos quedaremos con la primera observación de cada especie. Ahora filtramos de esta tabla las que tengan fecha de observación mayor a hoy menos 30 días (vistas en los últimos 30 días).

Esas serán las especies nuevas observadas en los últimos 30 días.

Primero haz el proceso y luego lo conviertes a una función. Es decir, carga el df_obs y haz los pasos con él, cuando te haya salido ya lo conviertes en función.

In [ ]:
def get_new_species(df_obs, last_days=30):
    """
    Obtiene las nuevas especies observadas en los últimos días.
    :param df_obs: DataFrame con las observaciones
    :param last_days: Número de días para considerar una especie como nueva
    :return: DataFrame con las nuevas especies
    """
    
    return df_new_species

Función para sacar una foto de las nuevas especies

In [ ]:
def get_photos_new_species(df_new_species, df_photos):
    # El dataframe df_new_species tiene las especies nuevas, con el id de cada observación
    # Filtramos el dataframe de fotos para quedarnos solo con las fotos de las especies nuevas, las de los ids de esas observaciones.
    # Puedes utilizar el método isin() de pandas para filtrar el dataframe df_photos
    # df_photos['id'].isin(df_new_species['id'])
    # Investiga el método isin() y cómo se utiliza para filtrar un dataframe
    # Nos quedaríamos solo con una foto para cada especie nueva, así que podemos usar el método drop_duplicates() de pandas con subset(['id'])
    
    return df_photos_new_species

Con esto estaríamos creando las funciones para extraer los datos. Luego estarían las de crear los gráficos y montar el informe.

# 5. Mapa calor para densidad de observaciones

Crear la función que toma un dataframe con el formato de df_obs (con esos nombres de columna, "latitude", "longitude") y lo mapee en el mapa de calor. Ese dataframe puede estar con las observaciones totales, filtrato por un kingdom, por un usuario, por un mes, o por lo que sea, pero no le afecta a la función, que lo hará siempre igual sobre un dataframe con las mismas columnas.

In [25]:
def plot_heatmap(df_obs):
    """
    Genera un mapa de calor de las observaciones.
    :param df_obs: DataFrame con las observaciones
    :return: None
    """
    
    return None
